
# PHÂN TÍCH DỮ LIỆU CHO THUÊ CĂN HỘ


## 1. Giới thiệu đề tài
### 1.1. Bối cảnh và lý do chọn đề tài
Dữ liệu căn hộ cho thuê chứa nhiều thông tin như giá thuê, số phòng ngủ, số phòng tắm và vị trí. Việc phân tích dữ liệu giúp khám phá đặc điểm các căn hộ và mối quan hệ giữa giá thuê và những thuộc tính liên quan

Đề tài sử dụng bộ dữ liệu Apartments for Rent Classified ARC từ UCI Machine Learning Reposity nhằm thực hành quy trình phân tích dữ liệu bằng ipynb, bao gồm khám phá, làm sạch, thống kê, trực quan hóa và áp dụng các thuật toán.

### 1.2. Mục tiêu
- Tìm hiểu cấu trúc và các thuộc tính của bộ dữ liệu căn hộ cho thuê
- Kiểm tra, làm sạch và xử lý các vấn đề trong dữ liệu
- Thực hiện thống kê mô tả và trực quan hóa để tìm hiểu đặc điểm, sự phân bố và mối quan hệ giữa các thuộc tính
- Áp dụng các phương pháp thống kê và thuật toán cơ bản đã học
- Nhận xét, đánh giá kết quả phân tích và rút ra kết luận từ dữ liệu

## 2. Tổng quan và khám phá dữ liệu
### 2.1. Giới thiệu bộ dữ liệu Apartment for Rent
Bộ dữ liệu Apartment for Rent hay chính xác hơn là Apartment for Rent Classified là một bộ dữ liệu và các tin cho thuê nhà và căn hộ tại Mỹ. Bộ dữ liệu trên được em lấy từ UCI, một trang web có mã nguồn dataset mở, được đóng góp vào 25/12/2019. Bộ dữ liệu có 22 thuộc tính và khoảng 100000 bản ghi.

Với AFRC, em tập trung tìm hiểu đăc điểm của các căn hộ cho thuê, phân tích sự phân bố giá thuê và mối quan hệ giữa giá thuê với các thuộc tính liên quan. 

Nguồn: [AFRC Dataset](https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified)

### 2.2. Mô tả cấu trúc dữ liệu
Dataset gồm 22 thuộc tính, chia thành bốn nhóm chính.
1. Thông tin đăng ký: id, category, title, body, source, time
2. Thông tin căn hộ: square_feet, bedrooms, bathrooms, amenities, pets_allowed, has_photo
3. Thông tin giá thuê: price, price_display, price_type, currency, fee
4. Thông tin vị trí: address, cityname, state, latitude, longtitude

Các thuộc tính có cả kiểu dữ liệu số và chuỗi. Price sẽ là thuộc tính chính để tìm hiểu mối quan hệ giữa giá thuê với diện tích, số phòng, vị trí,...

### 2.3. Sơ đồ thực thể – liên kết (ERD)
Với dataset AFRC, em xây dựng ERD rồi 3 thực thể
1. THONGTINCANHO: Lưu thông tin về căn hộ
2. DIADIEM: Lưu thông tin thành phố hoặc địa điểm của căn hộ
3. NGUONTIN: Lưu thông tin nguồn đăng tin của căn hộ

Thực thể được liên kết thông qua PK và FK với hai mỗi quan hệ 1-n

![ERD của AFRC](../image/erdAFRC.png.png)

Giải thích:
Quan hệ 1: DIADIEM - THONGTINCANHO 1 - 0..n
- Một địa điểm có thể có nhiều tin đăng cho thuê
- Mỗi tin đăng thuộc một địa điểm trong mô hình.
- Liên kết bằng location_id

Quan hệ 2:
- Mọi nguồn đăng tin có thể cung cấp nhiều tin được đăng
- Mỗi tin đăng được gắn với một nguồn
- Liên kết bằng source_id

Em có bổ sung thêm 3 thuộc tính là listing_key, location_key và source_id để làm thuộc tính định danh và chỉ tồn tại trên ERD, không có sẵn trong dataset gốc.

### 2.4. Nạp dữ liệu vào Pandas
Sử dụng thư viện để đọc dataset từ file CSV và hiển thị vài dòng dữ liệu đầu để kiểm tra dữ liệu đã được nạp.

In [2]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/apartments_for_rent_classified_100K.csv",
    sep=";",
    encoding="cp1252",
    low_memory=False
)

display(df.head())

,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1.0,1.0,USD,No,Thumbnail,...,"$2,195",Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1577360355
1,5668639818,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3.0,USD,No,Thumbnail,...,"$1,250",Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1577360340
2,5668639686,housing/rent/apartment,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2.0,3.0,USD,No,Thumbnail,...,"$1,395",Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1577360332
3,5668639659,housing/rent/apartment,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1.0,2.0,USD,No,Thumbnail,...,"$1,600",Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1577360330
4,5668639374,housing/rent/apartment,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1.0,1.0,USD,No,Thumbnail,...,$975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1577360308


**Nhận xét**: Dữ liệu đã được đẩy vào DataFrame thành công. Kêt quả hiển thị 5 dòng đầu tiên với 22 thuộc tính bao gồm thông tin đăng, giá thuê,... .

### 2.5. Khám phá thông tin tổng quan
Sau khi đã có dữ liệu, kiểm tra kích thước, tên thuộc tính, kiểu dữ liệu và các thông tin thống kê cơ bản của dữ liệu.

#### Kiểm tra kích thước và tên thuộc tính

In [3]:
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("Danh sách thuộc tính:")
print(df.columns.tolist())

Số dòng: 99492
Số cột: 22
Danh sách thuộc tính:
['id', 'category', 'title', 'body', 'amenities', 'bathrooms', 'bedrooms', 'currency', 'fee', 'has_photo', 'pets_allowed', 'price', 'price_display', 'price_type', 'square_feet', 'address', 'cityname', 'state', 'latitude', 'longitude', 'source', 'time']


#### Xem cấu trúc và kiểu dữ liệu

In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 99492 entries, 0 to 99491
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             99492 non-null  int64  
 1   category       99492 non-null  str    
 2   title          99492 non-null  str    
 3   body           99492 non-null  str    
 4   amenities      83448 non-null  str    
 5   bathrooms      99429 non-null  float64
 6   bedrooms       99368 non-null  float64
 7   currency       99492 non-null  str    
 8   fee            99492 non-null  str    
 9   has_photo      99492 non-null  str    
 10  pets_allowed   39068 non-null  str    
 11  price          99491 non-null  float64
 12  price_display  99491 non-null  str    
 13  price_type     99492 non-null  str    
 14  square_feet    99492 non-null  int64  
 15  address        7943 non-null   str    
 16  cityname       99190 non-null  str    
 17  state          99190 non-null  str    
 18  latitude       99

#### Thống kê cơ bản của dữ liệu

In [5]:
display(df.describe())

,id,bathrooms,bedrooms,price,square_feet,latitude,longitude,time
count,9.949200e+04,99429.000000,99368.000000,99491.000000,99492.000000,99467.000000,99467.000000,9.949200e+04
mean,5.358321e+09,1.445323,1.728212,1527.057281,956.430688,36.947988,-91.568656,1.559665e+09
std,1.847404e+08,0.547021,0.749200,904.245882,417.571522,4.599461,15.817168,1.105077e+07
min,5.121046e+09,1.000000,0.000000,100.000000,101.000000,19.573800,-159.369800,1.544174e+09
25%,5.197950e+09,1.000000,1.000000,1013.000000,729.000000,33.746500,-104.791900,1.550832e+09
50%,5.508673e+09,1.000000,2.000000,1350.000000,900.000000,37.228200,-84.562300,1.568745e+09
75%,5.509007e+09,2.000000,2.000000,1795.000000,1115.000000,39.953000,-77.608200,1.568767e+09
max,5.669439e+09,9.000000,9.000000,52500.000000,50000.000000,64.833200,-68.778800,1.577391e+09


**Nhận xét:**
- Bộ dữ liệu có 99429 bản ghi và 22 thuộc tính, gồm 5 thuộc tính float64, 3 thuộc tính int64 và 14 thuộc tính str
- Một số thuộc tính còn giá trị thiếu như amenities, pets_allowed, address, bathrooms, bedrooms, và vị trí, trong đó address và pets_allowed có số lượng giá trị thiếu khá lớn.
- Giá thuê (price) trung bình khoảng 1.527 USD, trung vị là 1.350 USD. Giá thuê thấp nhất là 100 USD và cao nhất là 52.500 USD.
- Diện tích (square_feet) trung bình khoảng 956 ft^2, trung vị là 900 ft^2 . Số phòng ngủ trung bình khoảng 1,73 và số phòng tắm trung bình khoảng 1,45.
- Từ ở phần thống kê cơ bản có thể thấy có một sự chênh lệch lớn của tiền thuê giữa căn hộ có diện tích lớn và nhỏ

### 2.6. Truy vấn dữ liệu có điều kiện
Sử dụng các lệnh để lọc và sắp xếp dữ liệu theo một số điều kiện nhằm xem thông tin các tin đăng thỏa mãn yêu cầu.

#### Lọc theo một điều kiện
Hiển thị các tin đăng có 2 phòng ngủ.

In [6]:
df_2_bedrooms = df.loc[
    df["bedrooms"] == 2,
    ["id", "cityname", "state", "bedrooms", "square_feet", "price"]
]

print("Số tin đăng có 2 phòng ngủ:", len(df_2_bedrooms))
display(df_2_bedrooms.head(10))

Số tin đăng có 2 phòng ngủ: 45975


,id,cityname,state,bedrooms,square_feet,price
3,5668639659,Vacaville,CA,2.0,820,1600.0
5,5668639368,Albuquerque,NM,2.0,965,1250.0
6,5668638765,Hampton,VA,2.0,1120,1600.0
7,5668638578,Lakewood,CO,2.0,947,1300.0
8,5668638476,Charleston,WV,2.0,600,795.0
9,5668638122,Chatsworth,CA,2.0,1005,2150.0
11,5668637998,Marblehead,MA,2.0,3077,3195.0
12,5668637809,Washington,DC,2.0,896,2395.0
15,5668636381,Kirkland,WA,2.0,872,2000.0
19,5668633909,Portland,OR,2.0,435,1250.0


#### Lọc thoe nhiều điều kiện
Tìm các tin đăng có giá thuê từ 1000 USD đến 2000 USD trên một tháng

In [7]:
df_price_range = df.loc[
    (df["price"] >= 1000) &
    (df["price"] <= 2000) &
    (df["currency"] == "USD") &
    (df["price_type"] == "Monthly"),
    ["id", "cityname", "state", "price", "currency",
     "price_type", "square_feet"]
]

print("Số tin đăng thỏa điều kiện:", len(df_price_range))
display(df_price_range.head(10))

Số tin đăng thỏa điều kiện: 57968


,id,cityname,state,price,currency,price_type,square_feet
1,5668639818,Newport News,VA,1250.0,USD,Monthly,1500
2,5668639686,Raleigh,NC,1395.0,USD,Monthly,1650
3,5668639659,Vacaville,CA,1600.0,USD,Monthly,820
5,5668639368,Albuquerque,NM,1250.0,USD,Monthly,965
6,5668638765,Hampton,VA,1600.0,USD,Monthly,1120
7,5668638578,Lakewood,CO,1300.0,USD,Monthly,947
10,5668638015,Alpharetta,GA,1795.0,USD,Monthly,1522
15,5668636381,Kirkland,WA,2000.0,USD,Monthly,872
17,5668635018,Villa Park,IL,1045.0,USD,Monthly,700
18,5668634255,Oxford,MS,1800.0,USD,Monthly,1500


#### Sắp xếp dữ liệu
Sắp xếp các tin đã lọc theo giá thuê bên trên theo giảm dần

In [8]:
df_sorted = df_price_range.sort_values(
    by="price",
    ascending=False
)

display(df_sorted.head(10))

,id,cityname,state,price,currency,price_type,square_feet
99328,5122057036,Suwanee,GA,2000.0,USD,Monthly,2394
37197,5508842276,Braintree,MA,2000.0,USD,Monthly,1000
72,5668639731,Kent,WA,2000.0,USD,Monthly,1220
15,5668636381,Kirkland,WA,2000.0,USD,Monthly,872
58594,5198520256,Irvine,CA,2000.0,USD,Monthly,737
58589,5198521971,Naples,FL,2000.0,USD,Monthly,768
15187,5508715410,Harrison,NJ,2000.0,USD,Monthly,785
58564,5198531516,Beverly,MA,2000.0,USD,Monthly,716
15149,5508741312,Boston,MA,2000.0,USD,Monthly,1000
15137,5508747893,Boston,MA,2000.0,USD,Monthly,1000


**Nhận xét:**
1. Có 45975 tin có 2 phòng ngủ. Trong 10 dòng được hiển thị, các căn hộ có diện tích và giá thuê khác nhau tùy theo từng tin đăng
2. Có 57968 tin thỏa mãn giá thuê từ 1000 đến 2000 USD một thnags. CHo thấy nhiều tin đăng nằm trong khoảng giá được chọn
3. Sau khi sắp xếp giá thuê giảm dần, 10 dòng đầu tiên có giá thuê 2000 USD. Tuy có cùng mức giá hưng cso thể thấy diện tích và vị trí là khác nhau

### 2.7. Sắp xếp và lọc dữ liệu
Tiếp tục lọc các tin đăng theo vị trí và đặc điểm căn hộ, sau đó sắp xếp kết quả theo nhiều thuộc tính để dễ quan sát và so sánh dữ liệu

#### Lọc theo vị trí và số phòng ngủ

In [9]:
df_ca = df.loc[
    (df["state"] == "CA") &
    (df["bedrooms"] >= 2),
    ["id", "cityname", "state", "bedrooms", "square_feet", "price"]
]

print("Số tin đăng thỏa điều kiện:", len(df_ca))
display(df_ca.head(10))


Số tin đăng thỏa điều kiện: 5750


,id,cityname,state,bedrooms,square_feet,price
3,5668639659,Vacaville,CA,2.0,820,1600.0
9,5668638122,Chatsworth,CA,2.0,1005,2150.0
28,5668630836,Ladera Ranch,CA,3.0,1525,3250.0
39,5668627095,San Jose,CA,3.0,1739,3990.0
41,5668627015,Oakland,CA,2.0,860,2550.0
42,5668626906,Mission Viejo,CA,4.0,3350,4950.0
43,5668626824,Berkeley,CA,2.0,1200,4950.0
46,5668625042,Sherman Oaks,CA,2.0,1391,2548.0
61,5668615317,Sacramento,CA,2.0,572,1385.0
106,5668627070,Hawthorne,CA,2.0,1070,2200.0


#### Sắp xếp theo nhiều thuộc tính

In [10]:
df_ca_sorted = df_ca.sort_values(
    by=["bedrooms", "square_feet"],
    ascending=[True, False]
)

display(df_ca_sorted.head(10))

,id,cityname,state,bedrooms,square_feet,price
32210,5509261541,Pasadena,CA,2.0,2535,5750.0
17162,5508880768,Los Angeles,CA,2.0,2372,6400.0
39472,5508925329,Beverly Hills,CA,2.0,2330,11000.0
20827,5509043710,Santa Monica,CA,2.0,2328,5950.0
5298,5664598216,Encinitas,CA,2.0,2283,5350.0
33700,5508681886,Los Angeles,CA,2.0,2274,7920.0
16174,5508798859,Los Angeles,CA,2.0,2248,6500.0
40698,5508905079,West Hollywood,CA,2.0,2246,11500.0
77742,5197985405,Indio,CA,2.0,2147,2250.0
75580,5198008022,Beaumont,CA,2.0,2127,1800.0


**Nhận xét:**
1. Có 5750 tin tại bang CA có từ 2 phòng ngủ trở lên. Trong 10 dòng đầu tiên, số phòng ngủ, diện tích, giá thuê có sự khác nhau tùy theo căn hộ
2. Sau khi sắp xếp theo số phòng ngủ tăng dần, diện tích giảm dần, 10 dòng đầu đều có 2 phòng ngủm diện tích từ 2100 đến 2550 ft^2. Giá thuê của các căn hộ này cũng có sự chênh lệch, từ 1800 đến 11500 USD.

## 3. Tiền xử lý và làm sạch dữ liệu
### 3.1. Kiểm tra dữ liệu thiếu
### 3.2. Trực quan hóa dữ liệu thiếu
### 3.3. Kiểm tra dữ liệu trùng lặp
### 3.4. Kiểm tra kiểu dữ liệu và định dạng
### 3.5. Phát hiện giá trị bất thường
### 3.6. Xử lý dữ liệu và kiểm tra kết quả

## 4. Thống kê mô tả và trực quan hóa dữ liệu
### 4.1. Các đại lượng đo xu hướng trung tâm
### 4.2. Các đại lượng đo độ phân tán
### 4.3. Độ lệch và độ nhọn của phân phối
### 4.4. Các biểu đồ thống kê dữ liệu

## 5. Phân phối xác suất
### 5.1. Phân phối nhị thức (Binomial)
### 5.2. Phân phối Poisson

## 6. Kiểm định giả thuyết thống kê
### 6.1. Kiểm định T-Test
### 6.2. Kiểm định Z-Test

## 7. Hiệp phương sai, tương quan và hồi quy tuyến tính
### 7.1. Hiệp phương sai (Covariance)
### 7.2. Hệ số tương quan (Correlation)
### 7.3. Hồi quy tuyến tính (Linear Regression)
### 7.4. Đánh giá mô hình hồi quy

## 8. Học máy có giám sát
### 8.1. Hồi quy Logistic (Logistic Regression)
### 8.2. Phân loại Naive Bayes

## 9. Phân cụm và luật kết hợp
### 9.1. Phân cụm K-Means
### 9.2. Luật kết hợp (Association Rules)

## 10. Cây quyết định và PCA
### 10.1. Cây quyết định (Decision Tree)
### 10.2. Phân tích thành phần chính (PCA)

## 11. Kết luận
### 11.1. Tổng hợp kết quả
### 11.2. Hạn chế